# 03a — Filtro de Liquidez (Pré-Integridade)

**Parte 1 de 3 do passo 3 do pipeline.** Executa os filtros de presença, IPO e ADTV e exporta
`data/Tickers/universo_pos_liquidez.csv` — a lista que o classificador de integridade irá avaliar.

**Não grava a matriz final.** Essa responsabilidade é do notebook `03_01c_Pos_Integridade.ipynb`,
que roda *após* o classificador ter gerado `data/Tickers/tickers_excluidos_integridade.csv`.

**Sequência obrigatória:**
1. `03_01a_Pre_Integridade.ipynb` ← este notebook
2. `11_Classificador_Integridade/Apendice_Classificador_Integridade_Universo_v2.ipynb`
3. `03_01c_Pos_Integridade.ipynb`

O orquestrador `run_pipeline_force.py` garante essa ordem automaticamente.

In [1]:
import sys
import os
import time
from pathlib import Path
import logging
import numpy as np
import pandas as pd

from utils.filtros import filtrar_presenca, filtrar_integridade_ipo, filtrar_adtv_formacao
from utils.config_loader import carregar_parametros

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] - %(message)s")

print("=== Reprodutibilidade Técnica — Versões do Ambiente ===")
print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")
print("=======================================================")

=== Reprodutibilidade Técnica — Versões do Ambiente ===
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Pandas: 3.0.3
NumPy:  2.4.6


In [2]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
PARENT_PATH       = Path.cwd().parent.parent
DIRETORIO_ORIGEM  = PARENT_PATH / "data" / "dados_economatica"
DIRETORIO_CACHE   = PARENT_PATH / "data" / "dados_economatica_consolidados"
DIRETORIO_DESTINO = PARENT_PATH / "data" / "Matriz_Precos"
DIR_TICKERS       = PARENT_PATH / "data" / "Tickers"

DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)
DIRETORIO_CACHE.mkdir(parents=True, exist_ok=True)
DIR_TICKERS.mkdir(parents=True, exist_ok=True)

# ── Parâmetros (config.json) ───────────────────────────────────────────────────
config             = carregar_parametros()
DATA_INICIO        = pd.to_datetime(config["DATA_INICIO"])
DATA_FIM           = pd.to_datetime(config["DATA_FIM"])
LIMIAR_PRESENCA    = config["LIMIAR_PRESENCA"]
ANO_FORMACAO_ADTV  = config["ANO_FORMACAO_ADTV"]
PERCENTIL_LIQUIDEZ = config["PERCENTIL_LIQUIDEZ"]
COL_VOLUME         = config["COL_VOLUME"]

print(f"[OK] Janela de Estudo: {DATA_INICIO.date()} a {DATA_FIM.date()} | presença ≥ {LIMIAR_PRESENCA:.0%}")
print(f"[OK] Exclusão ADTV: Ano {ANO_FORMACAO_ADTV} | Percentil de Corte: p{PERCENTIL_LIQUIDEZ*100:.0f}")
print(f"[OK] Exportará universo pós-liquidez em: {DIR_TICKERS / 'universo_pos_liquidez.csv'}")

[OK] Janela de Estudo: 2010-01-01 a 2025-12-31 | presença ≥ 95%
[OK] Exclusão ADTV: Ano 2010 | Percentil de Corte: p10
[OK] Exportará universo pós-liquidez em: C:\VSCodeWorkspace\1_TCC_Final\data\Tickers\universo_pos_liquidez.csv


In [3]:
# ── Ingestão (Parquet com fallback CSV) ───────────────────────────────────────
caminho_parquet = DIRETORIO_CACHE / "dados_brutos_economatica.parquet"
caminho_csv     = DIRETORIO_CACHE / "dados_brutos_economatica.csv"

t_start = time.perf_counter()
if caminho_parquet.exists():
    print(f"[Cache] Carregando Parquet: {caminho_parquet.name}")
    df_consolidado = pd.read_parquet(caminho_parquet)
elif caminho_csv.exists():
    print(f"[Cache] Parquet não encontrado. Carregando CSV: {caminho_csv.name}")
    df_consolidado = pd.read_csv(
        caminho_csv, sep=";", header=[0, 1], index_col=0, parse_dates=True
    )
else:
    raise FileNotFoundError(
        f"Nenhum arquivo consolidado (Parquet ou CSV) em: {DIRETORIO_CACHE}"
    )

df_precos_brutos  = df_consolidado["Fechamento"]
df_volumes_brutos = df_consolidado["Volume$"]
t_end = time.perf_counter()

print(f"[Cache] Carregado em {t_end - t_start:.4f}s")
print(f"Preços brutos: {df_precos_brutos.shape} | Volumes brutos: {df_volumes_brutos.shape}")

[Cache] Carregando Parquet: dados_brutos_economatica.parquet


[Cache] Carregado em 0.2552s
Preços brutos: (6507, 496) | Volumes brutos: (6507, 496)


In [4]:
# ── Recorte temporal ──────────────────────────────────────────────────────────
df_precos  = df_precos_brutos.loc[DATA_INICIO:DATA_FIM]
df_volumes = df_volumes_brutos.loc[DATA_INICIO:DATA_FIM]
print(f"Preços na janela: {df_precos.shape} | NaNs: {df_precos.isna().sum().sum()/df_precos.size:.2%}")

Preços na janela: (3967, 496) | NaNs: 47.55%


In [5]:
# ── Saneamento de calendário ──────────────────────────────────────────────────
df_calendario  = df_precos.dropna(how="all")
total_pregoes  = df_calendario.shape[0]
df_volumes     = df_volumes.reindex(df_calendario.index)
removidas      = df_precos.shape[0] - total_pregoes
print(f"[Calendário] feriados removidos: {removidas} | pregões válidos: {total_pregoes}")

[Calendário] feriados removidos: 0 | pregões válidos: 3967


In [6]:
# ── Filtro de Presença ≥ 95% ──────────────────────────────────────────────────
proporcao_presenca, aprovados_presenca, reprovados_presenca = filtrar_presenca(
    df_calendario, LIMIAR_PRESENCA
)
df_liquidos = df_calendario[aprovados_presenca].copy()
print(
    f"[Presença ≥ {LIMIAR_PRESENCA:.0%}] aprovados: {len(aprovados_presenca)} | "
    f"reprovados: {len(reprovados_presenca)}"
)

[Presença ≥ 95%] aprovados: 135 | reprovados: 361


In [7]:
# ── Filtro de IPO tardio ──────────────────────────────────────────────────────
data_inicio_janela, ativos_integros, ativos_ipo_tardio = filtrar_integridade_ipo(
    df_liquidos
)
df_integros = df_liquidos[ativos_integros].copy()
print(
    f"[Integridade IPO] início: {data_inicio_janela.date()} | "
    f"íntegros: {len(ativos_integros)} | IPO tardio: {len(ativos_ipo_tardio)}"
)

[Integridade IPO] início: 2010-01-04 | íntegros: 131 | IPO tardio: 4


In [8]:
# ── Filtro de ADTV (ano de formação) ─────────────────────────────────────────
adtv, limiar_adtv, ativos_liquidos, ativos_iliquidos = filtrar_adtv_formacao(
    df_volumes, ativos_integros, ANO_FORMACAO_ADTV, PERCENTIL_LIQUIDEZ
)
df_pos_liquidez = df_integros[ativos_liquidos].copy()

print(f"[ADTV {ANO_FORMACAO_ADTV}] limiar (p{PERCENTIL_LIQUIDEZ*100:.0f}): R$ {limiar_adtv:,.2f}/dia")
print(f"Aprovados: {len(ativos_liquidos)} | Excluídos: {len(ativos_iliquidos)}")
print(f"ADTV mediano aprovados: R$ {adtv[ativos_liquidos].median():,.2f}/dia")

[ADTV 2010] limiar (p10): R$ 374,412.21/dia
Aprovados: 118 | Excluídos: 13
ADTV mediano aprovados: R$ 6,748,086.10/dia


In [9]:
# ── Exporta universo pós-liquidez para o Classificador ───────────────────────
#
# ATENÇÃO: este é o único output deste notebook.
# O Classificador de Integridade (03_01b) deve apontar ARQ_UNIVERSO para este arquivo.
# A matriz de preços final é gravada pelo notebook 03_01c_Pos_Integridade.ipynb.
#
arq_universo = DIR_TICKERS / "universo_pos_liquidez.csv"
(pd.Series(list(df_pos_liquidez.columns), name="ticker")
   .to_csv(arq_universo, index=False))

n = df_pos_liquidez.shape[1]
print("="*60)
print(f"[03a] Universo pós-liquidez exportado: {n} ativos")
print(f"      Arquivo: {arq_universo}")
print()
print("PRÓXIMO PASSO:")
print("  Execute o Classificador de Integridade (03_01b) com:")
print(f"    ARQ_UNIVERSO = {arq_universo}")
print("  Ele gerará: data/Tickers/tickers_excluidos_integridade.csv")
print("  Depois execute: 03_01c_Pos_Integridade.ipynb")
print("="*60)

# Persiste variáveis intermediárias para uso eventual em diagnóstico manual.
# O 03c carrega tudo novamente do parquet — não depende do estado em memória.
_funil = {
    "bruto": df_precos_brutos.shape[1],
    "presenca": len(aprovados_presenca),
    "ipo": len(ativos_integros),
    "adtv": len(ativos_liquidos),
}
print("\n[Funil parcial]")
for etapa, n in _funil.items():
    print(f"  {etapa:10s}: {n}")

[03a] Universo pós-liquidez exportado: 118 ativos
      Arquivo: C:\VSCodeWorkspace\1_TCC_Final\data\Tickers\universo_pos_liquidez.csv

PRÓXIMO PASSO:
  Execute o Classificador de Integridade (03_01b) com:
    ARQ_UNIVERSO = C:\VSCodeWorkspace\1_TCC_Final\data\Tickers\universo_pos_liquidez.csv
  Ele gerará: data/Tickers/tickers_excluidos_integridade.csv
  Depois execute: 03_01c_Pos_Integridade.ipynb

[Funil parcial]
  bruto     : 496
  presenca  : 135
  ipo       : 131
  adtv      : 118


# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Aplicação no NB 03_01a (Pré-Liquidez) |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | ✅ | Explica a ingestão, filtros de presença (95%), IPO tardio e cálculo de liquidez ADTV (p10). |
| 2 | Documentar o processo | ✅ | Racional das escolhas de corte e períodos de formação descritos em Markdown. |
| 3 | Divisão clara de células | ✅ | Células curtas e modulares com explicações prévias das equações e critérios de corte. |
| 4 | Modularizar código | ✅ | Importa funções utilitárias e parâmetros via config_loader. |
| 5 | Registrar dependências | ✅ | Dependências controladas em requirements.txt. |
| 6 | Controle de versão | ✅ | Arquivo sob versionamento Git. |
| 7 | Construir um pipeline | ✅ | Primeiro passo do estágio de liquidez, exportando universo_pos_liquidez.csv. |
| 8 | Compartilhar/explicar dados | ✅ | Declara o uso de cotações Economática (proprietárias) e define o dicionário de variáveis. |
| 9 | Ler, rodar e explorar | ✅ | Executável de ponta a ponta sem erros. |
| 10 | Pesquisa aberta | ✅ | Código disponibilizado abertamente no repositório. |
